# مقارنة النطق كلمة بكلمة بين المعلم والطالبهذا المفكرة يوضح كيفية مقارنة نطق الطالب بالنسبة إلى نطق المعلم لنص عربي مشكول.## الخطوات الرئيسية:1. تحميل النص العربي مع التشكيل2. تحميل تسجيلات المعلم والطالب3. تقسيم التسجيل إلى مقاطع كلمة بكلمة4. محاذاة التسجيلين باستخدام Dynamic Time Warping (DTW)5. استخراج خصائص الصوت (MFCC) لكل كلمة6. حساب درجة التشابه بين نطق المعلم والطالب7. عرض النتائج والتوصيات

## ١. استيراد المكتبات اللازمة

In [ ]:
import numpy as npimport librosaimport pandas as pdimport matplotlib.pyplot as pltimport warningsfrom sklearn.preprocessing import StandardScalerfrom IPython.display import Audio, display# إخفاء التحذيرات لجعل المخرجات أنظفwarnings.filterwarnings('ignore')# إعدادات الرسوم البيانيةplt.rcParams['figure.figsize'] = (14, 6)# استخدام خط يدعم العربيةplt.rcParams['font.family'] = 'Noto Naskh Arabic'# دالة مساعدة لعرض النص العربي في المخططاتdef ar_text(text):    """تحسين عرض النص العربي في matplotlib"""    try:        import arabic_reshaper        from bidi.algorithm import get_display        reshaped = arabic_reshaper.reshape(text)        return get_display(reshaped)    except ImportError:        return textplt.rcParams['font.size'] = 12

> **ملاحظة حول عرض النص العربي:**>> إذا كانت المخططات لا تعرض النص العربي بشكل صحيح (حروف منفصلة)، يمكنك تثبيت المكتبات التالية:> ```bash> pip install arabic-reshaper python-bidi> ```

## ٢. تحميل النص العربي

In [ ]:
# النص العربي مع التشكيلarabic_text = (    "أَقْبَلَ فَصْلُ الرَّبِيعِ بِحُلَّتِهِ الزَّاهِيَةِ، فَازْدَانَتِ الأَرْضُ بِثَوْبٍ أَخْضَرَ جَمِيلٍ. "    "تَفَتَّحَتِ الزُّهُورُ المُلَوَّنَةُ فِي البَسَاتِينِ، وَانْتَشَرَتْ رَائِحَتُهَا الذَّكِيَّةُ فِي كُلِّ مَكَانٍ. "    "فِي الصَّبَاحِ البَاكِرِ، تُغَرِّدُ الطُّيُورُ عَلَى أَغْصَانِ الأَشْجَارِ كَأَنَّهَا تَعْزِفُ لَحْنًا لِلأَمَلِ. "    "وَتُرْسِلُ الشَّمْسُ أَشِعَّتَهَا الذَّهَبِيَّةَ لِتَنْشُرَ الدِّفْءَ وَالنُّورَ بَيْنَ الرَّوَابِي. "    "يَخْرُجُ الأَطْفَالُ إِلَى الحُقُولِ الواسِعَةِ، يَرْكُضُونَ وَيَمْرَحُونَ بِفَرَحٍ، فَمَا أَجْمَلَ الطَّبِيعَةَ حِينَ تَبْتَسِمُ لَنَا فِي هَذَا الفَصْلِ البَدِيعِ!")# تقسيم النص إلى كلماتwords = arabic_text.split()num_words = len(words)print(f"عدد الكلمات في النص: {num_words}")print("\nأول ١٠ كلمات:")for i, w in enumerate(words[:10], 1):    print(f"  {i}. {w}")

## ٣. تحميل تسجيلات الصوت

In [ ]:
# تحميل التسجيلات بمعدل أخذ عينات موحدsr = 16000y_teacher, sr = librosa.load('teacher.wav', sr=sr)y_student, _ = librosa.load('student.wav', sr=sr)print(f"مدة تسجيل المعلم: {len(y_teacher) / sr:.2f} ثانية")print(f"مدة تسجيل الطالب: {len(y_student) / sr:.2f} ثانية")print(f"معدل الأخذ: {sr} هرتز")# عرض مشغل الصوتprint("\n--- تسجيل المعلم ---")display(Audio(data=y_teacher, rate=sr))print("--- تسجيل الطالب ---")display(Audio(data=y_student, rate=sr))

## ٤. تقسيم الصوت إلى مقاطع كلمة بكلمةنستخدم طريقة التقسيم التناسبي: نحسب نسبة عدد أحرف كل كلمة إلى إجمالي عدد أحرف النص، ونستخدم هذه النسبة لتقسيم الزمن الكلي للتسجيل.> **ملاحظة:** هذه الطريقة بسيطة وفعّالة لأن الكلمات الأطول تستغرق عادةً وقتاً أطول في النطق.

In [ ]:
# حساب عدد الأحرف لكل كلمة (مع التشكيل)char_counts = [len(w) for w in words]total_chars = sum(char_counts)proportions = [c / total_chars for c in char_counts]# حساب حدود كل كلمة في تسجيل المعلمteacher_duration = len(y_teacher) / srteacher_boundaries = [0]current_time = 0for p in proportions:    current_time += p * teacher_duration    teacher_boundaries.append(current_time)# حساب حدود كل كلمة في تسجيل الطالبstudent_duration = len(y_student) / srstudent_boundaries = [0]current_time = 0for p in proportions:    current_time += p * student_duration    student_boundaries.append(current_time)print("مثال على الحدود الزمنية لأول ٥ كلمات:")print(f"{'الكلمة':<20} {'بداية المعلم':>12} {'نهاية المعلم':>12} {'المدة':>8}")print("-" * 60)for i in range(5):    t_start = teacher_boundaries[i]    t_end = teacher_boundaries[i + 1]    dur = t_end - t_start    print(f"{words[i]:<20} {t_start:>12.2f} {t_end:>12.2f} {dur:>8.2f}")

## ٥. محاذاة التسجيلين باستخدام Dynamic Time Warping (DTW)يتحدث الطالب والمعلم بنفس النص، لكن بوتيرة مختلفة. خوارزمية DTW تسمح لنا بمحاذاة إطارات الصوت بين التسجيلين حتى لو كانا مختلفين في السرعة.بعد حساب المسار المحاذي، يمكننا تحويل أي زمن في تسجيل المعلم إلى الزمن المقابل في تسجيل الطالب.

In [ ]:
# استخراج خصائص MFCC للتسجيل الكاملn_mfcc = 13hop_length = 512teacher_mfcc = librosa.feature.mfcc(y=y_teacher, sr=sr, n_mfcc=n_mfcc, hop_length=hop_length)student_mfcc = librosa.feature.mfcc(y=y_student, sr=sr, n_mfcc=n_mfcc, hop_length=hop_length)print(f"أبعاد MFCC للمعلم: {teacher_mfcc.shape}")print(f"أبعاد MFCC للطالب: {student_mfcc.shape}")# توحيد المعايير (Standardization)scaler_t = StandardScaler()scaler_s = StandardScaler()teacher_mfcc_norm = scaler_t.fit_transform(teacher_mfcc.T).Tstudent_mfcc_norm = scaler_s.fit_transform(student_mfcc.T).T# حساب DTW بين التسجيلينD, wp = librosa.sequence.dtw(X=teacher_mfcc_norm, Y=student_mfcc_norm, metric='euclidean')# عكس المسار ليبدأ من البدايةwp = wp[::-1]teacher_frames = wp[:, 0]student_frames = wp[:, 1]print(f"\nالمسافة الكلية لـ DTW: {D[-1, -1]:.2f}")print(f"طول مسار المحاذاة: {wp.shape[0]} نقطة")# دالة لتحويل زمن المعلم إلى زمن الطالبdef teacher_time_to_student_time(t_teacher):    frame = int(t_teacher * sr / hop_length)    idx = np.searchsorted(teacher_frames, frame)    if idx >= len(teacher_frames):        idx = len(teacher_frames) - 1    return student_frames[idx] * hop_length / sr# تحديث حدود الطالب باستخدام DTWstudent_boundaries_dtw = [teacher_time_to_student_time(b) for b in teacher_boundaries]print("\nمثال على نتيجة المحاذاة لأول ٥ كلمات:")print(f"{'الكلمة':<20} {'مدة المعلم':>12} {'مدة الطالب':>12} {'الفرق':>8}")print("-" * 60)for i in range(5):    t_dur = teacher_boundaries[i + 1] - teacher_boundaries[i]    s_dur = student_boundaries_dtw[i + 1] - student_boundaries_dtw[i]    diff = s_dur - t_dur    print(f"{words[i]:<20} {t_dur:>12.2f} {s_dur:>12.2f} {diff:>+8.2f}")

## ٦. مقارنة النطق لكل كلمةلكل كلمة:1. نستخرج المقطع الصوتي من تسجيل المعلم والطالب2. نحسب خصائص MFCC للمقطعين3. نحسب مسافة DTW بين المقطعين4. نحول المسافة إلى درجة تشابه بين ٠ و ١٠٠

In [ ]:
results = []for i, word in enumerate(words):    # الحدود الزمنية    t_start = teacher_boundaries[i]    t_end = teacher_boundaries[i + 1]    s_start = student_boundaries_dtw[i]    s_end = student_boundaries_dtw[i + 1]        # تحويل إلى عينات    t_start_sample = int(t_start * sr)    t_end_sample = int(t_end * sr)    s_start_sample = int(s_start * sr)    s_end_sample = int(s_end * sr)        # استخراج المقاطع الصوتية    t_audio = y_teacher[t_start_sample:t_end_sample]    s_audio = y_student[s_start_sample:s_end_sample]        t_dur = t_end - t_start    s_dur = s_end - s_start        # تخطي المقاطع القصيرة جداً    if len(t_audio) < 512 or len(s_audio) < 512:        results.append({            'الكلمة': word,            'الرقم': i + 1,            'مدة_المعلم': round(t_dur, 2),            'مدة_الطالب': round(s_dur, 2),            'فرق_المدة': round(s_dur - t_dur, 2),            'درجة_التشابه': None,            'مسافة_DTW': None        })        continue        # تحديد n_fft ديناميكياً لتجنب التحذيرات    n_fft = min(2048, len(t_audio), len(s_audio))    if n_fft < 512:        n_fft = 512        # استخراج MFCC لكل مقطع    t_mfcc = librosa.feature.mfcc(y=t_audio, sr=sr, n_mfcc=n_mfcc,                                   hop_length=hop_length, n_fft=n_fft)    s_mfcc = librosa.feature.mfcc(y=s_audio, sr=sr, n_mfcc=n_mfcc,                                   hop_length=hop_length, n_fft=n_fft)        # توحيد المعايير    scaler_w = StandardScaler()    t_mfcc_norm = scaler_w.fit_transform(t_mfcc.T).T    s_mfcc_norm = scaler_w.fit_transform(s_mfcc.T).T        # حساب DTW للكلمة    D_w, _ = librosa.sequence.dtw(X=t_mfcc_norm, Y=s_mfcc_norm, metric='euclidean')    dtw_dist = D_w[-1, -1]        # تطبيع المسافة بطول المسار    path_len = max(t_mfcc.shape[1], s_mfcc.shape[1])    norm_dist = dtw_dist / path_len if path_len > 0 else 0        # تحويل المسافة إلى درجة تشابه (٠ إلى ١٠٠)    similarity = 100 * np.exp(-norm_dist / 10)        results.append({        'الكلمة': word,        'الرقم': i + 1,        'مدة_المعلم': round(t_dur, 2),        'مدة_الطالب': round(s_dur, 2),        'فرق_المدة': round(s_dur - t_dur, 2),        'درجة_التشابه': round(similarity, 1),        'مسافة_DTW': round(norm_dist, 2)    })# إنشاء جدول النتائجdf = pd.DataFrame(results)print("اكتملت المقارنة!\n")print(f"متوسط درجة التشابه الكلي: {df['درجة_التشابه'].mean():.1f} / 100")print(f"عدد الكلمات المحلولة: {df['درجة_التشابه'].notna().sum()} من {len(df)}")

## ٧. رسم المخططات البيانية

In [ ]:
# إنشاء ٣ مخططات بيانيةfig, axes = plt.subplots(3, 1, figsize=(16, 14))# --- المخطط ١: درجات التشابه لكل كلمة ---ax1 = axes[0]colors = ['#e74c3c' if s < 65 else '#f39c12' if s < 75 else '#27ae60'          for s in df['درجة_التشابه'].fillna(50)]bars = ax1.bar(range(len(df)), df['درجة_التشابه'], color=colors, edgecolor='black', linewidth=0.3)ax1.axhline(y=df['درجة_التشابه'].mean(), color='blue', linestyle='--',            label=f"المتوسط: {df['درجة_التشابه'].mean():.1f}")ax1.set_xlabel('رقم الكلمة')ax1.set_ylabel('درجة التشابه (من 100)')ax1.set_title('درجة تشابه النطق لكل كلمة بين المعلم والطالب')ax1.set_ylim(0, 100)ax1.legend()ax1.grid(axis='y', alpha=0.3)# --- المخطط ٢: مقارنة المدة ---ax2 = axes[1]x = np.arange(len(df))width = 0.35bars1 = ax2.bar(x - width/2, df['مدة_المعلم'], width, label='المعلم', color='#3498db')bars2 = ax2.bar(x + width/2, df['مدة_الطالب'], width, label='الطالب', color='#e67e22')ax2.set_xlabel('رقم الكلمة')ax2.set_ylabel('المدة (ثانية)')ax2.set_title('مقارنة مدة نطق كل كلمة')ax2.legend()ax2.grid(axis='y', alpha=0.3)# --- المخطط ٣: فرق المدة ---ax3 = axes[2]diffs = df['فرق_المدة'].valuescolors3 = ['#e74c3c' if d > 0.15 else '#3498db' if d < -0.15 else '#95a5a6' for d in diffs]ax3.bar(range(len(df)), diffs, color=colors3, edgecolor='black', linewidth=0.3)ax3.axhline(y=0, color='black', linestyle='-', linewidth=0.8)ax3.set_xlabel('رقم الكلمة')ax3.set_ylabel('فرق المدة (ثانية)')ax3.set_title('فرق مدة النطق (طالب - معلم) - القيم الموجبة تعني أن الطالب أبطأ')ax3.grid(axis='y', alpha=0.3)plt.tight_layout()plt.savefig('comparison_plots.png', dpi=150, bbox_inches='tight')plt.show()print("تم حفظ المخططات في ملف comparison_plots.png")

## ٨. عرض النتائج التفصيلية

In [ ]:
# عرض الجدول الكاملpd.set_option('display.max_rows', None)pd.set_option('display.width', None)print("=" * 80)print("جدول النتائج الكامل")print("=" * 80)display(df)

## ٩. الكلمات التي تحتاج تحسيناً

In [ ]:
# الكلمات الأقل تشابهاًprint("\n" + "=" * 80)print("أقل ١٥ كلمة تشابهاً (تحتاج ممارسة)")print("=" * 80 + "\n")worst_words = df.nsmallest(15, 'درجة_التشابه')[['الكلمة', 'درجة_التشابه',                                                    'مدة_المعلم', 'مدة_الطالب',                                                    'فرق_المدة']]display(worst_words)

## ١٠. رسم خريطة حرارية لمسافات DTW

In [ ]:
# رسم خريطة حرارية لمسافات DTWfig, ax = plt.subplots(figsize=(18, 4))dtw_values = df['مسافة_DTW'].values.reshape(1, -1)im = ax.imshow(dtw_values, aspect='auto', cmap='YlOrRd', interpolation='nearest')ax.set_xticks(range(len(df)))ax.set_xticklabels([w[:8] for w in df['الكلمة']], rotation=90, fontsize=8)ax.set_yticks([])ax.set_title('خريطة حرارية لمسافات DTW لكل كلمة (الأحمر = مسافة أكبر = تشابه أقل)')plt.colorbar(im, ax=ax, label='مسافة DTW')plt.tight_layout()plt.show()

## ١١. مقارنة مقاطع صوتية محددةيمكنك هنا الاستماع إلى مقاطع صوتية محددة للمقارنة المباشرة.

In [ ]:
# دالة لاستخراج مقطع صوتي لكلمة محددةdef compare_word_audio(word_index, num_words_context=0):    """    استخراج وعرض مقطع صوتي لكلمة محددة مع سياقها.        المعاملات:        word_index: رقم الكلمة (يبدأ من ١)        num_words_context: عدد الكلمات الإضافية قبل وبعد    """    idx = word_index - 1    start_idx = max(0, idx - num_words_context)    end_idx = min(len(words), idx + num_words_context + 1)        t_start = teacher_boundaries[start_idx]    t_end = teacher_boundaries[end_idx]    s_start = student_boundaries_dtw[start_idx]    s_end = student_boundaries_dtw[end_idx]        t_audio_seg = y_teacher[int(t_start * sr):int(t_end * sr)]    s_audio_seg = y_student[int(s_start * sr):int(s_end * sr)]        context_words = ' '.join(words[start_idx:end_idx])    print(f"الكلمات: {context_words}")    print(f"\nالمعلم:")    display(Audio(data=t_audio_seg, rate=sr))    print(f"الطالب:")    display(Audio(data=s_audio_seg, rate=sr))# مقارنة أسوأ ٣ كلماتworst_3 = df.nsmallest(3, 'درجة_التشابه')['الرقم'].tolist()for idx in worst_3:    word = df[df['الرقم'] == idx]['الكلمة'].values[0]    score = df[df['الرقم'] == idx]['درجة_التشابه'].values[0]    print("\n" + "=" * 60)    print(f"مقارنة الكلمة رقم {idx}: '{word}' (درجة: {score})")    print("=" * 60)    compare_word_audio(idx, num_words_context=1)

## ١٢. التلخيص والتوصيات

In [ ]:
print("=" * 80)print("تقرير مقارنة النطق")print("=" * 80 + "\n")avg_score = df['درجة_التشابه'].mean()median_score = df['درجة_التشابه'].median()std_score = df['درجة_التشابه'].std()print(f"📊 الإحصائيات:")print(f"   متوسط درجة التشابه: {avg_score:.1f} / 100")print(f"   الوسيط: {median_score:.1f}")print(f"   الانحراف المعياري: {std_score:.1f}\n")# الكلمات الأبطأslowest = df.nlargest(5, 'فرق_المدة')[['الكلمة', 'فرق_المدة', 'درجة_التشابه']]print(f"🐢 الكلمات التي نطقها الطالب أبطأ من المعلم:")for _, row in slowest.iterrows():    print(f"   {row['الكلمة']}: +{row['فرق_المدة']:.2f} ثانية (درجة: {row['درجة_التشابه']})")# الكلمات الأسرعfastest = df.nsmallest(5, 'فرق_المدة')[['الكلمة', 'فرق_المدة', 'درجة_التشابه']]print(f"\n⚡ الكلمات التي نطقها الطالب أسرع من المعلم:")for _, row in fastest.iterrows():    print(f"   {row['الكلمة']}: {row['فرق_المدة']:.2f} ثانية (درجة: {row['درجة_التشابه']})")# التوصيةprint(f"\n💡 التوصية:")if avg_score >= 75:    print("   نطق ممتاز! يمكنك الانتقال إلى نص أصعب.")elif avg_score >= 65:    print("   نطق جيد، لكن ينصح بالممارسة على الكلمات الأقل من ٧٠ درجة.")else:    print("   يحتاج النطق إلى ممارسة مكثفة. ركز على الكلمات الأقل من ٦٥ درجة.")print("\n" + "=" * 80)